# AP Commander — GRPO Training Pipeline

**Environment:** `https://pathikreet-ap-clerk-env.hf.space` (HF Space, always running)  
**Compute:** This Colab notebook (T4 GPU)  
**Goal:** Verify the full GRPO pipeline works end-to-end in 1 epoch

```
Colab T4 GPU  ──(HTTP)──►  HF Space Environment
(model lives here)         (scorer lives here)
```

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — switch runtime to T4')
print('VRAM:', f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB" if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Go to Runtime > Change runtime type > T4 GPU'

In [ ]:
# Install dependencies — TRL standard stack (no Unsloth: avoids llm_blender conflicts)
!pip install -q --upgrade trl>=0.15.0 accelerate peft transformers bitsandbytes
!pip install -q requests datasets matplotlib
print('Done')

In [ ]:
import requests

ENV_URL = 'https://pathikreet-ap-clerk-env.hf.space'

# Verify the environment is reachable
health = requests.get(f'{ENV_URL}/health', timeout=30).json()
print('Environment status:', health['status'])
print('Version:', health.get('version'))
print('Total tasks:', health.get('total_tasks'))

tasks = requests.get(f'{ENV_URL}/tasks', timeout=30).json()
print(f'Tasks available: {len(tasks)}')
for t in tasks[:5]:
    print(f"  {t['task_id']} ({t['difficulty']})")
print('  ...')

In [ ]:
# Quick sanity check: one full episode manually
reset = requests.post(f'{ENV_URL}/reset',
                      json={'task_id': 'easy_perfect_match', 'seed': 42}).json()
session_id = reset['session_id']
obs = reset['observation']
print(f"Task: {obs['task_name']}")
print(f"Invoice total: ${obs['invoice']['invoice_total']:,.2f}")
print(f"Vendor: {obs['invoice']['vendor_name']}")

step = requests.post(f'{ENV_URL}/step', json={
    'session_id': session_id,
    'action': {
        'decision': 'APPROVE_FULL',
        'approved_amount': obs['invoice']['invoice_total'],
        'reason_code': 'MATCH_CONFIRMED',
        'explanation': f"Invoice matches PO and GRN. Total ${obs['invoice']['invoice_total']:.2f} approved."
    }
}).json()

print(f"Score: {step['reward']['score']}")
print(f"Feedback: {step['reward']['feedback']}")
print('Environment working correctly!')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# Qwen2.5-1.5B for fast pipeline test; swap to Qwen2.5-7B for real training
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
# MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'  # full training run

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

lora_cfg = LoraConfig(
    r=8, lora_alpha=8,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0, bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
print(f'Model loaded: {MODEL_NAME}')

In [ ]:
import json, re, random, collections

ENV_URL = 'https://pathikreet-ap-clerk-env.hf.space'

SYSTEM_PROMPT = """You are an AI Accounts Payable Clerk. Review the invoice, PO, and GRN, then output ONLY valid JSON:
{"decision": "APPROVE_FULL"|"APPROVE_PARTIAL"|"REJECT"|"ESCALATE"|"QUERY_VENDOR",
 "approved_amount": <float>,
 "reason_code": "MATCH_CONFIRMED"|"QUANTITY_MISMATCH"|"PRICE_DISCREPANCY"|"POLICY_VIOLATION"|"NO_PO_FOUND"|"DUPLICATE_INVOICE"|"VENDOR_MISMATCH"|"TAX_DISCREPANCY"|"PENDING_CLARIFICATION"|"MANAGER_REVIEW",
 "explanation": "<cite specific $ amounts>"}"""

VALID_DECISIONS    = {'APPROVE_FULL','APPROVE_PARTIAL','REJECT','ESCALATE','QUERY_VENDOR','HOLD'}
VALID_REASON_CODES = {'MATCH_CONFIRMED','QUANTITY_MISMATCH','PRICE_DISCREPANCY','POLICY_VIOLATION',
                      'NO_PO_FOUND','DUPLICATE_INVOICE','VENDOR_MISMATCH','TAX_DISCREPANCY',
                      'PENDING_CLARIFICATION','MANAGER_REVIEW'}

def obs_to_prompt(obs):
    inv = obs['invoice']
    lines = '\n'.join(f"  {li['description']}: qty={li['quantity']}, unit_price=${li['unit_price']:.2f}"
                      for li in inv.get('line_items', []))
    pos = '\n'.join(
        f"  PO {p['po_number']} ({p['status']}) {p['vendor_name']}: " +
        ', '.join(f"{l['description']} qty={l['ordered_quantity']} @${l['agreed_unit_price']:.2f}"
                  for l in p.get('lines', []))
        for p in obs.get('purchase_orders', []))
    grns = '\n'.join(
        f"  GRN {g['grn_id']}: " + ', '.join(f"{l['description']} recv={l['received_quantity']}"
                                              for l in g.get('lines', []))
        for g in obs.get('goods_receipts', []))
    context = '\n'.join(f'  {n}' for n in obs.get('context_notes', []))
    paid = ', '.join(obs.get('paid_invoice_ids', []))
    return (f"TASK: {obs['task_name']}\n{obs['task_description']}\n\n"
            f"INVOICE {inv['invoice_id']} | {inv['vendor_name']} | ${inv['invoice_total']:,.2f}\n{lines}\n"
            f"Freight: ${inv.get('freight_charge',0):.2f}\n\n"
            f"PURCHASE ORDERS:\n{pos}\n\nGOODS RECEIPTS:\n{grns}\n"
            + (f"PAID LEDGER: {paid}\n" if paid else "")
            + (f"CONTEXT:\n{context}\n" if context else "")
            + f"\nPOLICY:\n{obs['company_policy']}\n\nOutput JSON decision.")

def parse_action(raw):
    clean = re.sub(r'```(?:json)?\s*|\s*```', '', raw).strip()
    m = re.search(r'\{.*\}', clean, re.DOTALL)
    if m:
        try:
            a = json.loads(m.group())
            if (a.get('decision') in VALID_DECISIONS and
                a.get('reason_code') in VALID_REASON_CODES and
                isinstance(a.get('approved_amount'), (int, float)) and
                len(a.get('explanation', '')) > 10):
                return a, True
        except Exception:
            pass
    return {'decision': 'REJECT', 'approved_amount': 0.0,
            'reason_code': 'NO_PO_FOUND', 'explanation': 'parse error fallback'}, False

def _greedy_followup(obs_dict):
    """Scripted intermediate-step policy for accumulated episode rollouts."""
    notes = ' '.join(obs_dict.get('context_notes', [])).lower()
    total = abs(float(obs_dict.get('invoice', {}).get('invoice_total', 0) or 0))
    if any(k in notes for k in ('manager approved', 'vp approved', 'cfo approved', 'approved by')):
        return {'decision': 'APPROVE_FULL', 'approved_amount': total,
                'reason_code': 'MATCH_CONFIRMED', 'explanation': f'Approval confirmed. Approving ${total:.2f}.'}
    if any(k in notes for k in ('fraudulent', 'duplicate', 'already paid', 'deny')):
        return {'decision': 'REJECT', 'approved_amount': 0.0,
                'reason_code': 'DUPLICATE_INVOICE', 'explanation': 'Confirmed fraud/duplicate. Rejecting.'}
    if any(k in notes for k in ('flagged', 'violation', 'sox', 'non-compliant')):
        return {'decision': 'REJECT', 'approved_amount': 0.0,
                'reason_code': 'POLICY_VIOLATION', 'explanation': 'Compliance violation. Rejecting.'}
    return {'decision': 'REJECT', 'approved_amount': 0.0,
            'reason_code': 'PENDING_CLARIFICATION', 'explanation': 'Could not resolve. Rejecting for safety.'}

def run_episode_accumulated(task_id, first_action, seed=None, discount=0.9, max_steps=5):
    """
    Runs full multi-step episode with discounted reward accumulation.
    QUERY_VENDOR→REJECT = 0.01 + 0.9*0.99 = 0.901 > shortcut REJECT = ~0.4
    """
    try:
        r = requests.post(f'{ENV_URL}/reset', json={'task_id': task_id, 'seed': seed}, timeout=20)
        session_id = r.json()['session_id']
        action = first_action
        total = 0.0
        for step_n in range(max_steps):
            result = requests.post(f'{ENV_URL}/step',
                                   json={'session_id': session_id, 'action': action},
                                   timeout=20).json()
            total += (discount ** step_n) * float(result['reward']['score'])
            if result['done']:
                break
            action = _greedy_followup(result['observation'])
        return min(0.99, max(0.01, total))
    except Exception:
        return 0.01

# ── Curriculum sampler ──────────────────────────────────────────────────────
_TASK_DIFFICULTY = {
    'easy_perfect_match': 'easy',      'easy_no_po_found': 'easy',
    'medium_quantity_shortfall':'medium','medium_price_discrepancy':'medium',
    'medium_split_delivery':'medium',   'medium_vendor_mismatch':'medium',
    'hard_policy_violation':'hard',     'hard_duplicate_invoice':'hard',
    'hard_partial_po_match':'hard',     'hard_tax_discrepancy':'hard',
}
_UNLOCK_THRESHOLDS = {'easy': 0.70, 'medium': 0.65}
_DIFFICULTY_ORDER  = ['easy', 'medium', 'hard']

class CurriculumSampler:
    def __init__(self):
        self._rewards = collections.defaultdict(list)
        self.unlocked = {'easy'}
    def record(self, tid, r):
        self._rewards[tid].append(r)
        self._try_unlock()
    def mean_for(self, diff):
        v = [r for tid, d in _TASK_DIFFICULTY.items() if d == diff
             for r in self._rewards.get(tid, [])]
        return sum(v)/len(v) if v else 0.0
    def _try_unlock(self):
        for i, d in enumerate(_DIFFICULTY_ORDER[:-1]):
            if d in self.unlocked and self.mean_for(d) >= _UNLOCK_THRESHOLDS.get(d, 0.70):
                nxt = _DIFFICULTY_ORDER[i+1]
                if nxt not in self.unlocked:
                    self.unlocked.add(nxt)
                    print(f'[CURRICULUM] Unlocked {nxt}! mean({d})={self.mean_for(d):.3f}')
    def gate_task(self, tid):
        if _TASK_DIFFICULTY.get(tid, 'easy') in self.unlocked:
            return tid
        return random.choice([t for t, d in _TASK_DIFFICULTY.items() if d == 'easy'])
    def status(self):
        return ' | '.join(f'{d}={self.mean_for(d):.2f}{"✓" if d in self.unlocked else "✗"}'
                          for d in _DIFFICULTY_ORDER)

CURRICULUM = CurriculumSampler()
print('Helpers ready. Curriculum initialized (easy unlocked).')

In [ ]:
from datasets import Dataset

# Tasks for 1-epoch pipeline test (keep small)
TRAIN_TASKS = [
    'easy_perfect_match',
    'easy_no_po_found',
    'medium_quantity_shortfall',
    'medium_price_discrepancy',
    'hard_policy_violation',
    'hard_duplicate_invoice',
]

# Build prompts dataset
rows = []
for task_id in TRAIN_TASKS:
    for seed in [1, 2, 3]:  # 3 seeds per task = 18 samples total
        try:
            reset = requests.post(f'{ENV_URL}/reset',
                                  json={'task_id': task_id, 'seed': seed},
                                  timeout=15).json()
            obs = reset['observation']
            prompt_text = obs_to_prompt(obs)
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': prompt_text},
            ]
            rows.append({
                'prompt': tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                ),
                'task_id': task_id,
                'seed':    seed,
            })
        except Exception as e:
            print(f'  skip {task_id} seed={seed}: {e}')

dataset = Dataset.from_list(rows)
print(f'Dataset: {len(dataset)} samples')
print('Sample prompt (first 300 chars):')
print(dataset[0]['prompt'][:300])

In [ ]:
import torch

EVAL_TASKS = ['easy_perfect_match', 'easy_no_po_found',
              'medium_quantity_shortfall', 'medium_price_discrepancy',
              'hard_policy_violation', 'hard_duplicate_invoice']

def eval_task(task_id, seed=99):
    model.eval()
    reset = requests.post(f'{ENV_URL}/reset', json={'task_id': task_id, 'seed': seed}, timeout=20).json()
    obs, session_id = reset['observation'], reset['session_id']
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': obs_to_prompt(obs)}]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=250, temperature=0.1, do_sample=True)
    raw    = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    action, _ = parse_action(raw)
    score = float(requests.post(f'{ENV_URL}/step',
                                json={'session_id': session_id, 'action': action},
                                timeout=20).json()['reward']['score'])
    print(f'  {raw[:100].strip()}')
    return score

print('=== BASELINE (before training) ===')
baseline = {}
for t in EVAL_TASKS:
    s = eval_task(t)
    baseline[t] = s
    print(f'  {t}: {s:.3f}')
print(f'  Mean: {sum(baseline.values())/len(baseline):.3f}')
model.train()

In [ ]:
# Two independent reward functions (guide: separate env signal from format signal)

def env_reward_fn(completions, task_id=None, seed=None, **kwargs):
    """Accumulated discounted per-step reward from AP Commander environment."""
    task_ids = task_id if task_id else ['easy_perfect_match'] * len(completions)
    seeds    = seed    if seed    else [random.randint(1, 999)] * len(completions)
    rewards  = []
    for completion, tid, s in zip(completions, task_ids, seeds):
        gated = CURRICULUM.gate_task(tid)
        action, _ = parse_action(completion)
        score = run_episode_accumulated(gated, action, seed=int(s))
        CURRICULUM.record(gated, score)
        rewards.append(score)
    print(f'  curriculum: {CURRICULUM.status()}')
    return rewards

def format_reward_fn(completions, **kwargs):
    """Format reward: +0.05 valid JSON with correct fields, -0.05 otherwise."""
    return [0.05 if parse_action(c)[1] else -0.05 for c in completions]

# Smoke test
test = env_reward_fn(
    ['{"decision": "APPROVE_FULL", "approved_amount": 100.0, "reason_code": "MATCH_CONFIRMED", "explanation": "Invoice $100.00 matches PO and GRN exactly."}'],
    task_id=['easy_perfect_match'], seed=[42]
)
print(f'env_reward smoke test: {test[0]:.3f}')
print(f'format_reward smoke test: {format_reward_fn(["bad json"])[0]}  (should be -0.05)')

In [ ]:
from trl import GRPOConfig, GRPOTrainer
model.train()

NUM_GENERATIONS = 4  # completions per prompt (GRPO group size)

# per_device_train_batch_size MUST equal num_generations (TRL divisibility requirement)
config = GRPOConfig(
    output_dir='./ap_commander_grpo',
    num_train_epochs=1,
    per_device_train_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    max_completion_length=250,
    temperature=0.9,
    logging_steps=1,
    save_steps=999,
    report_to='none',
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[env_reward_fn, format_reward_fn],  # two independent signals
    args=config,
    train_dataset=dataset,
)

print(f'Training {len(dataset)} samples, 1 epoch, {NUM_GENERATIONS} gen/prompt')
result = trainer.train()
print(f'Done. Loss: {result.training_loss:.4f}')

In [ ]:
# Scores AFTER training
FastLanguageModel.for_inference(model)

print('=== POST-TRAINING ===')
post = {}
for t in EVAL_TASKS:
    score, raw = eval_task(t)
    post[t] = score
    print(f'  {t}: {score:.3f}  |  {raw[:80]}')
print(f'  Mean: {sum(post.values())/len(post):.3f}')

print('\n=== COMPARISON ===')
for t in EVAL_TASKS:
    delta = post[t] - baseline[t]
    arrow = '▲' if delta > 0 else ('▼' if delta < 0 else '=')
    print(f'  {t:<35} {baseline[t]:.3f} → {post[t]:.3f}  {arrow} {delta:+.3f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tasks  = list(EVAL_TASKS)
before = [baseline[t] for t in tasks]
after  = [post[t]     for t in tasks]
x      = np.arange(len(tasks))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - 0.2, before, 0.4, label='Before (1-epoch GRPO)', color='#e74c3c', alpha=0.8)
ax.bar(x + 0.2, after,  0.4, label='After',                 color='#2ecc71', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([t.replace('_', '\n') for t in tasks], fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score (0.01 – 0.99)')
ax.set_title('AP Commander — GRPO Pipeline Test (1 Epoch)')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4, label='0.5 baseline')
ax.legend()
plt.tight_layout()
plt.savefig('pipeline_test_results.png', dpi=120)
plt.show()
print('Saved: pipeline_test_results.png')

## What This Notebook Runs

**Stack**: TRL ≥ 0.15.0 standard (no Unsloth — dropped due to llm_blender dependency conflict)

**Key design decisions synced with `training/train.py`**:

| Decision | Why |
|---|---|
| `per_device_train_batch_size = num_generations` | TRL requires `generation_batch_size % num_generations == 0` — setting equal guarantees it |
| Two reward functions `[env_reward_fn, format_reward_fn]` | Guide: separate env signal from format signal, don't combine into one scalar |
| `run_episode_accumulated()` with discount=0.9 | QUERY_VENDOR→REJECT = 0.01 + 0.9×0.99 = 0.901 > shortcut REJECT ≈ 0.4, incentivizes multi-step |
| `CurriculumSampler` with gating | Easy unlocked at start; medium unlocks when easy mean ≥ 0.70; hard unlocks at medium ≥ 0.65 |
| Save LoRA adapters directly | Guide point 16: don't merge 4-bit naively — save adapters to `Pathikreet/ap-commander-adapter` |

**For the A10G full run** (HF Space): swap `MODEL_NAME` to `Qwen/Qwen2.5-7B-Instruct`, `NUM_GENERATIONS=8`, `num_train_epochs=3`.